In [ ]:
from google.colab import drive
import os
import shutil

# 1. Google Drive'ı Bağla
drive.mount('/content/drive')

# 2. Drive'dan yedeklenen dosyaları yerel disk alanına kopyala
print("Yedek dosyaları Google Drive'dan indiriliyor...")
shutil.copy('/content/drive/My Drive/opencv_cuda/cv2_cuda_backup.zip', '/content/')
shutil.copy('/content/drive/My Drive/opencv_cuda/opencv_libs_backup.zip', '/content/')

# 3. Python paket klasörünü oluştur ve dosyaları çıkart
print("Python cv2 paketi kuruluyor...")
os.makedirs('/usr/local/lib/python3.13/dist-packages/cv2', exist_ok=True)
!unzip -q -o /content/cv2_cuda_backup.zip -d /usr/local/lib/python3.13/dist-packages/cv2

# 4. Paylaşılan C++ kütüphanelerini çıkart
print("Sistem paylaşımlı kütüphaneleri (.so) geri yükleniyor...")
!unzip -q -o /content/opencv_libs_backup.zip -d /usr/local/lib

# 5. Sistem dinamik kütüphane bağlayıcısını güncelle
print("Kütüphane yolları sistemde güncelleniyor...")
!ldconfig

# 6. Test Et
print("\n=== KURULUM TAMAMLANDI ===")
import cv2
cuda_devices = cv2.cuda.getCudaEnabledDeviceCount()
print(f"CUDA Destekli Cihaz Sayısı: {cuda_devices}")
if cuda_devices > 0:
    print("Tebrikler! OpenCV CUDA desteğiyle başarıyla geri yüklendi.")
    cv2.cuda.printCudaDeviceInfo(0)
else:
    print("Uyarı: Kurulum tamamlandı ancak GPU etkinleştirilmemiş olabilir. Runtime ayarlarından GPU seçtiğinize emin olun.")

Mounted at /content/drive
Yedek dosyaları Google Drive'dan indiriliyor...
Python cv2 paketi kuruluyor...
Sistem paylaşımlı kütüphaneleri (.so) geri yükleniyor...
Kütüphane yolları sistemde güncelleniyor...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libjulia.so.1.12 is not a sym

In [2]:
import cv2
from time import perf_counter_ns

input_path = "resim1 (2).jpeg"

# Fotoğrafı aç
photo = cv2.imread(input_path, cv2.IMREAD_COLOR)

if photo is None:
    raise FileNotFoundError(
        f"{input_path} bulunamadı. Dosya adını ve uzantısını kontrol et."
    )

# 8K UHD boyutları
target_width = 7680
target_height = 4320

# Fotoğrafın oranını koruyarak 8K alanı dolduracak şekilde büyüt
height, width = photo.shape[:2]
scale = max(target_width / width, target_height / height)

new_width = int(width * scale)
new_height = int(height * scale)

resized = cv2.resize(
    photo,
    (new_width, new_height),
    interpolation=cv2.INTER_LINEAR
)

# Fazla kısımları ortadan kırpıp tam 7680×4320 yap
x_start = (new_width - target_width) // 2
y_start = (new_height - target_height) // 2

image_8k = resized[
    y_start:y_start + target_height,
    x_start:x_start + target_width
]

# 8K 3 kanallı resmi BMP olarak kaydet
cv2.imwrite("resim1_8k.bmp", image_8k)

# BMP'yi OpenCV ile aç
image = cv2.imread("resim1_8k.bmp", cv2.IMREAD_COLOR)

if image is None:
    raise FileNotFoundError("resim1_8k.bmp açılamadı.")

# Yalnızca pikselleri 2'ye bölme işleminin süresini ölç
start = perf_counter_ns()
edited_image = image // 2
end = perf_counter_ns()

elapsed_ns = end - start

# Düzenlenmiş resmi BMP olarak kaydet
cv2.imwrite("resim1_duzenlenmis_8k.bmp", edited_image)

print(f"Çözünürlük: {image.shape[1]} × {image.shape[0]}")
print(f"Kanal sayısı: {image.shape[2]}")
print(f"Düzenleme süresi: {elapsed_ns:,} ns")
print(f"Düzenleme süresi: {elapsed_ns / 1_000_000:.3f} ms")
print(f"Düzenleme süresi: {elapsed_ns / 1_000_000_000:.6f} saniye")

Çözünürlük: 7680 × 4320
Kanal sayısı: 3
Düzenleme süresi: 36,119,587 ns
Düzenleme süresi: 36.120 ms
Düzenleme süresi: 0.036120 saniye
